# 第三章：交易成本 / Chapter 3: Trading Costs

## 你将学到 / What You Will Learn
- 每笔交易要付哪些钱 / The different fees you pay per trade
- 什么是点差（买卖之间的"差价"）/ What the spread is (the gap between bid and ask)
- 什么是手续费 / What commission is
- 什么是隔夜利息（持仓过夜的代价）/ What swap is (the cost of holding overnight)

---

## 3.1 三种成本一句话说清 / 3.1 The Three Costs in One Line Each

| 成本 / Cost | 类比 / Analogy | 什么时候收 / When It Is Charged |
|------|------|-----------|
| **点差 / Spread** | 像换外币时银行的买卖差价 / Like the bank's buy / sell gap at the FX window | 每次开仓时自动扣 / Auto-deducted whenever you open a trade |
| **手续费 / Commission** | 像股票交易的佣金 / Like a stockbroker's commission | 开仓和平仓各收一次 / Charged once on entry and once on exit |
| **隔夜利息 / Swap** | 像银行贷款利息 / Like interest on a bank loan | 每天凌晨6点持仓就收 / Charged every morning at ~06:00 for open positions |

### 点差 = 买价和卖价之间的差 / Spread = Gap Between the Buy Price and the Sell Price

> 你去银行换美元：银行卖给你 7.25，你卖给银行 7.20。
> 这个差价 0.05 就是银行赚的钱。外汇交易的"点差"就是这个道理。
>
> Changing RMB at a bank: the bank sells USD to you at 7.25 and buys from you at 7.20. That 0.05 gap is the bank's profit. The FX "spread" is the exact same idea.

### 隔夜利息的特殊规则 / A Special Rule About Swap

> 周三持仓过夜 → 收 **3倍** 利息（因为包含了周六周日两天）
>
> Holding through Wednesday night → **3×** swap is charged (it packs Saturday and Sunday in).

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, ToggleButtons
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# Color palette
C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
print('Setup done!')

## 3.2 互动计算器 / 3.2 Interactive Cost Calculator

> 调整手数、点差、持仓天数，看看你的总成本是多少。
>
> Adjust the lot size, spread, and holding period to see your total cost.

In [ ]:
def cost_calculator(lots=1.0, spread=1.3, commission_per_lot=6.0,
                    swap_per_lot_per_day=-2.5, days=5):
    plt.close('all')
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

    # Calculations
    spread_cost = spread * 10 * lots  # 1 pip ~ $10/lot on major pairs
    commission = commission_per_lot * lots
    wed_count = days // 7 + (1 if days % 7 >= 3 else 0)
    swap_cost = abs(swap_per_lot_per_day) * lots * days + abs(swap_per_lot_per_day) * lots * 2 * wed_count
    total = spread_cost + commission + swap_cost

    # --- Left: pie chart ---
    ax = axes[0]
    sizes = [spread_cost, commission, swap_cost]
    labels = [f'Spread\n${spread_cost:.1f}', f'Commission\n${commission:.1f}', f'Swap\n${swap_cost:.1f}']
    ax.pie(sizes, labels=labels, colors=[C_BLUE, C_ORANGE, C_PURPLE],
           autopct='%1.0f%%', startangle=90, textprops={'fontsize': 10})
    ax.set_title(f'Total cost: ${total:.2f}', fontsize=14, fontweight='bold')

    # --- Middle: cost growth over time ---
    ax = axes[1]
    day_arr = np.arange(0, 31)
    spread_fix = np.full_like(day_arr, spread_cost, dtype=float)
    comm_fix = np.full_like(day_arr, commission, dtype=float)
    swap_cum = np.array([abs(swap_per_lot_per_day) * lots * d + abs(swap_per_lot_per_day) * lots * 2 * (d // 7 + (1 if d % 7 >= 3 else 0))
                         for d in day_arr])
    total_curve = spread_fix + comm_fix + swap_cum

    ax.fill_between(day_arr, 0, spread_fix, color=C_BLUE, alpha=0.3, label='Spread (once)')
    ax.fill_between(day_arr, spread_fix, spread_fix + comm_fix, color=C_ORANGE, alpha=0.3, label='Commission (once)')
    ax.fill_between(day_arr, spread_fix + comm_fix, total_curve, color=C_PURPLE, alpha=0.3, label='Swap (daily)')
    ax.plot(day_arr, total_curve, 'k-', lw=2)
    ax.axvline(x=days, color=C_RED, ls='--', alpha=0.7)
    ax.set_xlabel('Holding Days')
    ax.set_ylabel('Cumulative Cost (USD)')
    ax.set_title('Cost vs Time', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # --- Right: multiplier by weekday ---
    ax = axes[2]
    week_labels = ['Mon', 'Tue', 'Wed\n(x3!)', 'Thu', 'Fri', 'Sat\n(off)', 'Sun\n(off)']
    mults = [1, 1, 3, 1, 1, 0, 0]
    bar_colors = [C_BLUE] * 2 + [C_RED] + [C_BLUE] * 2 + ['#95a5a6'] * 2
    ax.bar(week_labels, mults, color=bar_colors, edgecolor='white', lw=1.5)
    ax.set_title('Swap Multiplier by Day', fontweight='bold')
    ax.set_ylabel('Multiplier')
    for i, v in enumerate(mults):
        if v > 0:
            ax.text(i, v + 0.1, f'{v}x', ha='center', fontweight='bold', fontsize=11)
    ax.set_ylim(0, 4)

    plt.tight_layout()
    plt.show()
    print(f'  Holding {days} days total cost: ${total:.2f}')
    print(f'  Breakdown: Spread ${spread_cost:.1f} + Commission ${commission:.1f} + Swap ${swap_cost:.1f}')
    if days > 7:
        print('  Note: longer holding -> swap dominates. Short-term is cheaper.')

interact(cost_calculator,
         lots=FloatSlider(value=1.0, min=0.1, max=10, step=0.1, description='Lots:'),
         spread=FloatSlider(value=1.3, min=0.1, max=10, step=0.1, description='Spread(pips):'),
         commission_per_lot=FloatSlider(value=6.0, min=0, max=20, step=1, description='Comm($/lot):'),
         swap_per_lot_per_day=FloatSlider(value=-2.5, min=-15, max=5, step=0.5, description='Swap($/lot/d):'),
         days=IntSlider(value=5, min=0, max=30, step=1, description='Days:'));